# QIGuard Stellar: Classical MLP & PennyLane VQC Hybrid Model Training

This notebook trains both:
1. **Soroban Classical MLP Encoder & Risk Scorer** (PyTorch, 8D -> 16D -> 8D -> 1D)
2. **Soroban PennyLane Variational Quantum Circuit (VQC)** (8-qubit parameter-shift circuit with RX angle embedding, parameterized RY/RZ rotations, and CZ entanglement)

> **Running in Google Colab / Colab VS Code Extension**:
> Connect this notebook to a Colab runtime (CPU or T4 GPU) using the VS Code Colab extension or [Google Colab](https://colab.research.google.com).

In [ ]:
# 1. Install dependencies
!pip install --quiet torch pennylane numpy

In [ ]:
import os
import sys
import json
import hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from typing import Dict, Any, Tuple
import pennylane as qml

print(f"PyTorch version: {torch.__version__}")
print(f"PennyLane version: {qml.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {device}")

In [ ]:
# 2. Model Definitions
class SorobanClassicalMLP(nn.Module):
    def __init__(self, input_dim: int = 8, hidden_dim: int = 16, latent_dim: int = 8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )
        self.classifier_head = nn.Sequential(
            nn.Linear(latent_dim, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        latent = self.encoder(x)
        logits = self.classifier_head(latent)
        return logits, latent

class SorobanPennyLaneHybrid(nn.Module):
    def __init__(self, classical_mlp: SorobanClassicalMLP, n_qubits: int = 8, circuit_depth: int = 4):
        super().__init__()
        self.classical_mlp = classical_mlp
        self.n_qubits = n_qubits
        self.circuit_depth = circuit_depth

        dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def quantum_circuit(inputs, weights_ry, weights_rz):
            for i in range(n_qubits):
                qml.RX(np.pi * inputs[i], wires=i)
            for l in range(circuit_depth):
                for i in range(n_qubits):
                    qml.RY(weights_ry[l, i], wires=i)
                    qml.RZ(weights_rz[l, i], wires=i)
                for i in range(n_qubits):
                    qml.CZ(wires=[i, (i + 1) % n_qubits])
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.qnode = quantum_circuit
        self.weights_ry = nn.Parameter(torch.randn(circuit_depth, n_qubits) * 0.1)
        self.weights_rz = nn.Parameter(torch.randn(circuit_depth, n_qubits) * 0.1)
        self.readout = nn.Sequential(
            nn.Linear(n_qubits, 4),
            nn.Tanh(),
            nn.Linear(4, 1),
            nn.Sigmoid()
        )

    def forward_quantum(self, latent: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        batch_size = latent.shape[0]
        exp_vals_list = []
        for b in range(batch_size):
            x_b = torch.clamp(latent[b], 0.0, 1.0)
            exp_z = torch.stack(self.qnode(x_b, self.weights_ry, self.weights_rz))
            exp_vals_list.append(exp_z)
        exp_vals = torch.stack(exp_vals_list)
        quantum_delta = 3.0 + (self.readout(exp_vals.float()) * 12.0)
        return quantum_delta, exp_vals

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        logits_classical, latent = self.classical_mlp(x)
        prob_classical = torch.sigmoid(logits_classical) * 100.0
        quantum_delta, exp_vals = self.forward_quantum(latent)
        hybrid_score = torch.clamp(prob_classical + quantum_delta, 0.0, 99.0)
        return hybrid_score, prob_classical, quantum_delta, exp_vals

In [ ]:
# 3. Execute Training Pipeline
# If running in local repo cloned in Colab:
data_path = "../dataset"
if not os.path.exists(data_path):
    data_path = "dataset"

from scripts.train_hybrid_models import load_data, train_classical, train_quantum_hybrid, main

if __name__ == "__main__":
    main()